## Testing Notebook for new additions to the data scraping process 

In [1]:
import pandas as pd 
import requests 
import numpy as np 
from bs4 import BeautifulSoup
import json, os, time, pdb 
import sys 
import warnings, logging 
import itertools 
from tqdm import tqdm
from argparse import ArgumentParser
from util_funcs import * 
from selenium import webdriver 
from selenium.webdriver.chrome.options import Options 
from selenium.webdriver.common.by import By 
from selenium.common.exceptions import NoSuchElementException, WebDriverException
from selenium.webdriver.support.wait import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC


def close_out_driver(wd):
    wd.close()
    wd.quit()


headers = {'User-Agent': 
           'Mozilla/5.0 (X11; Linux x86_64)'+\
            'AppleWebKit/537.36 (KHTML, like Gecko) Chrome/119.0.0.0 Safari/537.36'
}


## Driver startup

In [2]:
options = webdriver.ChromeOptions()
driver = webdriver.Chrome(options=options)

In [4]:
driver.close()
driver.quit()

## Part 1: Team Data pulls for specific seasons and leagues 

In [2]:
league = 180659 # Six Nations league code
season = 2025


# Below is equivalent to get_teams_list function in the scraper code 
test_url = 'https://www.espn.co.uk' + \
    '/rugby/table/_/league/{}/season/{}'.format(league, season)
response = requests.get(test_url, headers=headers).content
soup_team = BeautifulSoup(response, 'html.parser')

get_index = lambda x, char: x.find(char)

tbodies = soup_team.find_all('tbody')

team_name_link = {}
for tbody in tbodies:

    row_trs = tbody.find_all('tr')

    for tr in row_trs:

        td_start = tr.find_all('td')[0]
        try:
            team_link = td_start.find_all('a')[0]['href']
            
            team_name_link[td_start.find_all('a')[1].find('span').text] = [
                team_link,
                team_link[team_link.find('id/')+3:team_link.find('id/')+3+\
                            get_index(team_link[team_link.find('id/')+3:], '/')]
            ]
        except:
            team_link = None 
            team_name_link[td_start.find_all('span')[3].find_all('span')[0].text] = [
                team_link,
                None
            ]
print(team_name_link )

{'France': ['/rugby/team/_/id/9/france', '9'], 'England': ['/rugby/team/_/id/1/england', '1'], 'Ireland': ['/rugby/team/_/id/3/ireland', '3'], 'Scotland': ['/rugby/team/_/id/2/scotland', '2'], 'Italy': ['/rugby/team/_/id/20/italy', '20'], 'Wales': ['/rugby/team/_/id/4/wales', '4']}


In [3]:
team_link

'/rugby/team/_/id/4/wales'

In [4]:
team_name_link.keys()

dict_keys(['France', 'England', 'Ireland', 'Scotland', 'Italy', 'Wales'])

In [5]:
sched_dict = {}

results_base_url = "https://www.espn.co.uk/rugby/results/_/team/"
for team in team_name_link.keys():
    team_id = team_name_link[team][1]
    if team_id is not None:

        team_page_resp = requests.get(
            results_base_url+team_id+'/league/{}/season/{}'.format(league, season),
            headers=headers).content
        team_soup = BeautifulSoup(team_page_resp, 'html.parser')

        full_sched = team_soup.find(id='sched-container')
        match_months = full_sched.find_all('tbody')

        sched_dict[team] = match_months 

In [6]:
def parse_scheds(sched):
    
    totals = []
    for mon in sched:

        rows = mon.find_all('tr')

        for row in rows:

            first_row = row.find_all('td')
            date = first_row[0].text

            home_base, away_base = first_row[1].find_all('a')[0], \
                    first_row[2].find_all('a')[0]

            home_team, away_team = home_base.find('span').text, \
                    away_base.find('span').text

            home_team_abbr, away_team_abbr = home_base.find('abbr').text, \
                    away_base.find('abbr').text

            try:
                game_link = first_row[1].find_all('span')[-1].find('a')['href']
                game_id, league_id = game_link[
                    game_link.find('Id/')+3:game_link.find('/league')], \
                game_link[game_link.find('league/')+7:]
                score = first_row[1].find_all('span')[-1].find('a').text

            except TypeError:
                game_link, game_id, league_id = np.nan, np.nan, np.nan 
                score = first_row[1].find_all('span')[-1].text
    
            competition, stadium = first_row[4].text, first_row[5].text
            home_score, away_score = score.split()[0], score.split()[-1]

            totals.append([
                date, home_team, away_team, home_team_abbr,
                away_team_abbr, game_link, score, home_score,
                away_score, competition, stadium, game_id, league_id
            ])

    comb_df = pd.DataFrame(
        totals,
        columns=[
            'date', 'home_team', 'away_team', 
            'home_team_abbr', 'away_team_abbr', 'game_link', 
            'score', 'home_score', 'away_score', 
            'competition', 'stadium', 'game_id', 'league_id']
    )

    return comb_df 

In [7]:
comb_df = []
for team in sched_dict.keys():
    parsed_sched = parse_scheds(sched_dict[team])
    parsed_sched['season'] = season
    comb_df.append(parsed_sched)

team_dfs = pd.concat(comb_df, axis=0, ignore_index=True).drop_duplicates()

In [8]:
# Util function used in schedule pulls 
def label_table_parse(labels, table):
    row_step = []
    for row in table:
        val_text = row.text
        if val_text not in labels:
            row_step.append(val_text)

    row_ordered = [[row_step[i-1], row_step[i]] \
                    for i in range(1, len(row_step), 2)]

    return row_ordered 


# Takes the game and league to parse the match stats 
def get_match_stats(game_id, league_id): 

    # URL declaration, scraping, and initial parsing to get the tables on the page 
    url = 'https://www.espn.com/rugby/matchstats/_/gameId/{}/league/{}'.format(
        game_id, league_id
    )
    response = requests.get(url, headers=headers).content
    soup = BeautifulSoup(response, 'html.parser')
    tables = soup.find_all('table')

    # Group labels 
    group_2_labels = ['Tries', 'Conversion Goals', 
                        'Penalty Goals', 'Kick Percent Success'
    ]
    group_3_labels = ['Kicks From Hand', 'Passes', 'Runs']
    group_4_labels = [
        'Possession 1H/2H', 'Territory 1H/2H', 'Clean Breaks', 
        'Defenders Beaten', 'Offload', 'Rucks Won', 
        'Mauls Won', 'Turnovers Conceded'
    ]
    group_8_labels = ['Red Cards', 'Yellow Cards', 'Total Free Kicks Conceded']

    # Parsed data lists used by multiple groups 
    four_tables = soup.find_all(
        class_='sub-module equal-height countChartList height-reset'
    )
    check_top_largeLabels = soup.find_all(
        class_="stat-graph compareLineGraph twoTeam largeLabels"
    )
    stacked_rls = soup.find_all(class_='stacked-rl')

    # GROUP 1: Home and Away team parsing 
    top_bar = soup.find(class_='competitors')

    h_a_team_bar = [
        top_bar.find(class_='team team-a'),
        top_bar.find(class_='team team-b')
    ]

    # GROUP 2: Match Events 
    match_event = four_tables[0].find('tbody')
    match_event_rows = match_event.find_all('td')

    group_2_ordered = label_table_parse(group_2_labels, match_event_rows)

    # GROUP 3: Kick/Pass/Run 
    home_away_total_meters = [
        int(i.text) for i in check_top_largeLabels[0].find_all(class_='chartValue')
    ]

    meter_rows = four_tables[1].find('tbody').find_all('td')
    group_3_ordered = label_table_parse(group_3_labels, meter_rows)

    # GROUP 4: Attacking 
    attack_rows = stacked_rls[0].find('tbody').find_all('td')
    group_4_ordered = label_table_parse(group_4_labels, attack_rows)

    # GROUP 5: Possession and Territory 
    terr_vals = soup.find_all(
        class_="stat-graph compareLineGraph twoTeam largeLabels large"
    )[0].find_all(class_='chartValue')

    poss_vals = check_top_largeLabels[1].find_all(class_='chartValue')

    # GROUP 6: Set Pieces 
    sp_charts = four_tables[2].find_all(class_='countChart')

    # list of scrums and then lineouts. 00 is home scrums, 10 is home lineouts 
    h_a_set_pieces = [
        [
            sp_charts[0].find_all(class_='countLabel')[0].text,
            sp_charts[0].find_all(class_='countLabel')[1].text
        ],
        [
            sp_charts[1].find_all(class_='countLabel')[0].text,
            sp_charts[1].find_all(class_='countLabel')[1].text
        ]
    ]

    # GROUP 7: Defending 
    # list of lists, raw tackles and then home tackles 
    tackles = [
        four_tables[3].find_all(class_='home-team'),
        four_tables[3].find_all(class_='away-team')
    ]

    # GROUP 8: Discipline and Penalties 
    disc_rows = tables[3].find('tbody').find_all('td')
    penalty = stacked_rls[1].find(class_='countChart').find_all(
        class_='countLabel'
    )

    group_8_ordered = label_table_parse(group_8_labels, disc_rows)

    # Combine all group variables 
    top_data_dict = {
        'game_id': game_id,
        'league_id': league_id,

        # GROUP 1 metrics 
        'home_team': h_a_team_bar[0].find(class_='short-name').text,
        'home_team_score': int(h_a_team_bar[0].find(class_='score-container').text), 
        'away_team': h_a_team_bar[1].find(class_='short-name').text,
        'away_team_score': int(h_a_team_bar[1].find(class_='score-container').text),

        # GROUP 2 metrics 
        'home_tries': group_2_ordered[0][0],
        'away_tries': group_2_ordered[0][1],
        'home_conversions': group_2_ordered[1][0],
        'away_conversions': group_2_ordered[1][1],
        'home_penalty_goals': group_2_ordered[2][0],
        'away_penalty_goals': group_2_ordered[2][1],
        'home_kick_percent': group_2_ordered[3][0],
        'away_kick_percent': group_2_ordered[3][1],

        # GROUP 3 metrics 
        'home_total_meters': home_away_total_meters[0],
        'away_total_meters': home_away_total_meters[1],
        'home_kfh': group_3_ordered[0][0],
        'away_kfh': group_3_ordered[0][1],
        'home_pass_meters': group_3_ordered[1][0],
        'away_pass_meters': group_3_ordered[1][1],
        'home_runs': group_3_ordered[2][0],
        'away_runs': group_3_ordered[2][1],

        # GROUP 4 metrics 
        'home_possession_1h_2h': group_4_ordered[0][0],
        'home_territory_1h_2h': group_4_ordered[1][0],
        'home_clean_breaks': group_4_ordered[2][0],
        'home_defenders_beaten': group_4_ordered[3][0],
        'home_offloads': group_4_ordered[4][0],
        'home_rucks_won': group_4_ordered[5][0],
        'home_mauls_won': group_4_ordered[6][0],
        'home_turnovers_conceeded': group_4_ordered[7][0],
        'away_possession_1h_2h': group_4_ordered[0][1],
        'away_territory_1h_2h': group_4_ordered[1][1],
        'away_clean_breaks': group_4_ordered[2][1],
        'away_defenders_beaten': group_4_ordered[3][1],
        'away_offloads': group_4_ordered[4][1],
        'away_rucks_won': group_4_ordered[5][1],
        'away_mauls_won': group_4_ordered[6][1],
        'away_turnovers_conceeded': group_4_ordered[7][1],

        # GROUP 5 metrics 
        'home_total_possession': poss_vals[0].text,
        'home_total_territory': terr_vals[0].text,
        'away_total_possesion': poss_vals[1].text,
        'away_total_territory': terr_vals[1].text,

        # GROUP 6 metrics 
        'home_scrum': h_a_set_pieces[0][0],
        'home_lineout': h_a_set_pieces[1][0],
        'away_scrum': h_a_set_pieces[0][1],
        'away_lineout': h_a_set_pieces[1][1],

        # GROUP 7 metrics 
        'home_tackles': tackles[0][0].text, 
        'home_tackle_perc': tackles[1][0].text, 
        'away_tackles': tackles[0][1].text, 
        'away_tackle_perc': tackles[1][1].text,

        # GROUP 8 metrics 
        'home_red_cards': group_8_ordered[0][0],
        'home_yellow_cards': group_8_ordered[1][0],
        'home_free_kicks_con': group_8_ordered[2][0], 
        'away_red_cards': group_8_ordered[0][1],
        'away_yellow_cards': group_8_ordered[1][1],
        'away_free_kicks_con': group_8_ordered[2][1],
        'home_penalties': int(penalty[0].text),
        'away_penalties': int(penalty[1].text)

    }

    df = pd.DataFrame(top_data_dict, index=[0])
    
    return df 


In [11]:
def slice_prop_func(x, char):
    x = x.replace(" ", "")
    if char == '/':
        return int(x[:x.find(char)])
    elif char == '(':
            return int(x[x.find('/')+1:x.find('(')])  
    elif char == ')':
        return int(x[x.find('(')+1:x.find('%')]) * 0.01


def slice_poss_terr_func(x):
    x = x.replace(" ", "")
    two_h_val = int(x[x.find('/')+1:].replace('%', '')) * 0.01
    one_h_val = int(x[:x.find('/')].replace('%', '')) * 0.01
    return one_h_val, two_h_val


def clean_match_stats(df):
    
    perc_fix = lambda x: 0.0 if x == 'N/A' else int(x.replace("%", ""))*0.01
    
    prop_variable_names = ['rucks_won', 'mauls_won', 'scrum', 'lineout']
    for side in ['home', 'away']:
        try:
            df['{}_kick_percent'.format(side)] = df[
                '{}_kick_percent'.format(side)].apply(perc_fix)

            df['{}_1h_poss'.format(side)] = df[
                '{}_possession_1h_2h'.format(side)].apply(lambda x: slice_poss_terr_func(x)[0])
            df['{}_2h_poss'.format(side)] = df[
                '{}_possession_1h_2h'.format(side)].apply(lambda x: slice_poss_terr_func(x)[1])

            df['{}_1h_terr'.format(side)] = df[
                '{}_territory_1h_2h'.format(side)].apply(lambda x: slice_poss_terr_func(x)[0])
            df['{}_2h_terr'.format(side)] = df[
                '{}_territory_1h_2h'.format(side)].apply(lambda x: slice_poss_terr_func(x)[1])
        except:
            print('annoying error')
            pdb.set_trace()
        
        for var_group in prop_variable_names:
            
            if len(df[df['{}_{}'.format(side, var_group)].str.contains('NaN')]) > 0:
                print('stopped on annoying rows')
                pdb.set_trace()

            if 'won' in var_group:
                str_attach = ''
            else:
                str_attach = 'won_'

            df['{}_{}_{}'.format(side, var_group, 'num')] = \
                    df['{}_{}'.format(side, var_group)].apply(slice_prop_func, args=('/', ))
            df['{}_{}_{}'.format(side, var_group, 'total_num')] = \
                    df['{}_{}'.format(side, var_group)].apply(slice_prop_func, args=('(', ))
            df['{}_{}_{}'.format(side, var_group, str_attach+'percent')] = \
                    df['{}_{}'.format(side, var_group)].apply(slice_prop_func, args=(')', ))
            
    return df 

In [9]:
get_match_stats(
    game_id=team_dfs['game_id'].iloc[0],
    league_id=team_dfs['league_id'].iloc[0]
)

,game_id,league_id,home_team,home_team_score,away_team,away_team_score,home_tries,away_tries,home_conversions,away_conversions,...,away_tackles,away_tackle_perc,home_red_cards,home_yellow_cards,home_free_kicks_con,away_red_cards,away_yellow_cards,away_free_kicks_con,home_penalties,away_penalties
0,600264,180659,France,35,Scotland,16,4,1,3,1,...,123/138,89%,0,2,0,0,1,0,11,12


In [10]:
team_dfs

,date,home_team,away_team,home_team_abbr,away_team_abbr,game_link,score,home_score,away_score,competition,stadium,game_id,league_id,season
0,"Sat, Mar 15",France,Scotland,FRA,SCO,/rugby/match/_/gameId/600264/league/180659,35 - 16,35,16,Six Nations,"Stade de France, Saint-Denis",600264,180659,2025
1,"Sat, Mar 8",Ireland,France,IRE,FRA,/rugby/match/_/gameId/600259/league/180659,27 - 42,27,42,Six Nations,"Aviva Stadium, Dublin",600259,180659,2025
2,"Sun, Feb 23",Italy,France,ITA,FRA,/rugby/match/_/gameId/600258/league/180659,24 - 73,24,73,Six Nations,"Stadio Olimpico, Rome",600258,180659,2025
3,"Sat, Feb 8",England,France,ENG,FRA,/rugby/match/_/gameId/600254/league/180659,26 - 25,26,25,Six Nations,"Allianz Stadium, Twickenham, London",600254,180659,2025
4,"Fri, Jan 31",France,Wales,FRA,WAL,/rugby/match/_/gameId/600250/league/180659,43 - 0,43,0,Six Nations,"Stade de France, Saint-Denis",600250,180659,2025
5,"Sat, Mar 15",Wales,England,WAL,ENG,/rugby/match/_/gameId/600263/league/180659,14 - 68,14,68,Six Nations,"Principality Stadium, Cardiff",600263,180659,2025
6,"Sun, Mar 9",England,Italy,ENG,ITA,/rugby/match/_/gameId/600261/league/180659,47 - 24,47,24,Six Nations,"Allianz Stadium, Twickenham, London",600261,180659,2025
7,"Sat, Feb 22",England,Scotland,ENG,SCO,/rugby/match/_/gameId/600257/league/180659,16 - 15,16,15,Six Nations,"Allianz Stadium, Twickenham, London",600257,180659,2025
9,"Sat, Feb 1",Ireland,England,IRE,ENG,/rugby/match/_/gameId/600252/league/180659,27 - 22,27,22,Six Nations,"Aviva Stadium, Dublin",600252,180659,2025
10,"Sat, Mar 15",Italy,Ireland,ITA,IRE,/rugby/match/_/gameId/600262/league/180659,17 - 22,17,22,Six Nations,"Stadio Olimpico, Rome",600262,180659,2025


In [12]:
games_df = []

for game in tqdm(range(len(team_dfs))):

    try:
        if type(team_dfs['game_id'].iloc[game]) == type('tester'):
            games_df.append(get_match_stats(
                game_id=team_dfs['game_id'].iloc[game],
                league_id=team_dfs['league_id'].iloc[game],
            ))
    except Exception as e: 
        print(e)
        # pdb.set_trace()

100%|██████████| 15/15 [00:09<00:00,  1.61it/s]


In [18]:
team_df_join_back = team_dfs[['game_id', 'date', 'competition', 'season', 'stadium']]

In [20]:
all_teams_df = pd.concat(games_df, axis=0)
all_teams_df = all_teams_df.merge(team_df_join_back, how='left', on='game_id')
all_teams_df = clean_match_stats(all_teams_df)
all_teams_df.head()

,game_id,league_id,home_team,home_team_score,away_team,away_team_score,home_tries,away_tries,home_conversions,away_conversions,...,away_rucks_won_percent,away_mauls_won_num,away_mauls_won_total_num,away_mauls_won_percent,away_scrum_num,away_scrum_total_num,away_scrum_won_percent,away_lineout_num,away_lineout_total_num,away_lineout_won_percent
0,600264,180659,France,35,Scotland,16,4,1,3,1,...,0.96,5,5,1.00,5,7,0.71,15,16,0.93
1,600259,180659,Ireland,27,France,42,3,5,3,4,...,0.93,8,9,0.88,3,3,1.00,12,12,1.00
2,600258,180659,Italy,24,France,73,3,11,3,9,...,0.99,7,8,0.87,5,5,1.00,15,15,1.00
3,600254,180659,England,26,France,25,4,3,3,2,...,0.94,4,4,1.00,5,5,1.00,9,9,1.00
4,600250,180659,France,43,Wales,0,7,0,4,0,...,0.97,2,3,0.66,9,10,0.90,7,8,0.87


In [ ]:
# Internally used by get_player_stats function below 
def get_player_page_data(self, table, fields, team):
    
    group_df = [] 
    for player in range(1, len(table)):

        player_tag = table[player].find_elements(by=By.TAG_NAME, value='a')
        p_name, href_link = player_tag[0].text, player_tag[0].get_attribute('href')

        p_id = href_link[href_link.find('player/')+7:href_link.find('.html')]
        pos = table[player].find_elements(by=By.TAG_NAME, value='span')[0].text

        stats = [
            float(i.text) for i in table[player].find_elements(
                by=By.TAG_NAME, value='td')[1:]
        ]

        group_df.append([p_name, pos, p_id, team]+stats)
        
    titles = ['p_name', 'position', 'p_id', 'team'] + fields
    test_df = pd.DataFrame(group_df, columns=titles)
    
    return test_df 

# @staticmethod
def get_player_stats(self, game_id, league_id):
    
    fields = [
        ['tries', 'try_assists', 'conversion_goals',
            'penalty_goals', 'drop_goals_converted', 'points'],
        ['-', 'passes', 'runs', 'meters_run',
            'clean_breaks', 'defenders_beaten', 'offloads', '-'],
        ['to_conceded', 'tackles', 'missed_tackles', 'lineouts_won'],
        ['penalties_conceded', 'yellows', 'reds']
        ]
    
    page_url = "https://www.espn.co.uk/rugby/"+\
            "playerstats/_/gameId/{}/league/{}".format(game_id, league_id)
    self.driver.get(page_url)
    
    tab_labels = self.driver.find_element(
        by=By.CLASS_NAME, value='col-b').find_elements(
        by=By.TAG_NAME, value='div')[3]
    
    grouped_dfs = []
    for label in range(4):
        
        self.update_driver_to_page(tab_labels, label)
        
        group_table = self.driver.find_elements(by=By.TAG_NAME, value='table')
        home_group = group_table[0].find_elements(by=By.TAG_NAME, value='tr')
        away_group = group_table[1].find_elements(by=By.TAG_NAME, value='tr')
        pdb.set_trace()
        
        comb_df = pd.concat(
            [
                self.get_player_page_data(home_group, fields[label], 'home'),
                self.get_player_page_data(away_group, fields[label], 'away')
            ], axis=0
        )
        grouped_dfs.append(comb_df)
    # pdb.set_trace()
    final_df = pd.concat(grouped_dfs, axis=1)
    final_df['game_id'], final_df['league_df'] = game_id, league_id
    final_df = final_df.loc[:, ~final_df.columns.duplicated()]
    
    return final_df 
        

In [ ]:
def get_schedule(self, teams_list, league, season, team_input = None):
    
    if len(teams_list.keys()) == 0:
        sched_dict = {}
        
    elif team_input is not None:
        results_base_url = "https://www.espn.co.uk/rugby/results/_/team/"
        team_id = teams_list[team][1]

        team_page_resp = requests.get(
            results_base_url+team_id+'/league/{}/season/{}'.format(league, season), \
            headers=self.headers).content
        team_soup = BeautifulSoup(team_page_resp, 'html.parser')
        
        full_sched = team_soup.find(id='sched-container')
        match_months = full_sched.find_all('tbody')
    
    else:
        sched_dict = {}

        results_base_url = "https://www.espn.co.uk/rugby/results/_/team/"
        for team in teams_list.keys():
            team_id = teams_list[team][1]
            if team_id is not None:

                team_page_resp = requests.get(
                    results_base_url+team_id+'/league/{}/season/{}'.format(league, season),
                    headers=self.headers).content
                team_soup = BeautifulSoup(team_page_resp, 'html.parser')

                full_sched = team_soup.find(id='sched-container')
                match_months = full_sched.find_all('tbody')

                sched_dict[team] = match_months 

    return sched_dict